# Modelling

Baseline multi-class classification for `Irrigation_Need` (Low / Medium / High).

- Load preprocessed arrays from `data/processed/preprocessed_v1.1/`
- Evaluate several classical ML models with **Stratified K-Fold CV**
- Compare on `balanced_accuracy` (competition metric) and `f1_macro`
- Persist the best model to `models/`

## 1. Imports

In [1]:
import os
import pickle
import time
import warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from pred_irr_comp import MODELS_DIR, PROCESSED_DATA_DIR
from pred_irr_comp import LABEL_MAP, RAW_DATA_DIR
RANDOM_STATE = 42
warnings.filterwarnings('ignore')

2026-04-28 22:32:33.251 | INFO     | pred_irr_comp.config:<module>:10 - PROJ_ROOT path is: /Users/imaadh/Downloads/predicting-irr-kagglecomp


## 2. Load preprocessed data

Produced by `preprocessing.ipynb` (version `v1.1` — encoded, no feature engineering).

Produced by `preprocessing.ipynb` (version `v1.2` - encoded and with golden features)

Produced by `preprocessing.ipynb` (version `v1.3` - encoded and with golden features+ some boolean features)

In [2]:
DATA_VERSION = 'preprocessed_v1.3'
data_dir = PROCESSED_DATA_DIR / DATA_VERSION

X_train = pickle.load(open(data_dir / 'X_train.pkl', 'rb'))
y_train = pickle.load(open(data_dir / 'y_train.pkl', 'rb'))
X_test  = pickle.load(open(data_dir / 'X_test.pkl',  'rb'))

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape} | classes: {np.unique(y_train, return_counts=True)}')
print(f'X_test : {X_test.shape}')

X_train: (630000, 71)
y_train: (630000,) | classes: (array([0, 1, 2]), array([369917, 239074,  21009]))
X_test : (270000, 71)


## 3. Define models & CV setup

Stratified K-Fold (k=5) to preserve class balance across folds. Scoring: `balanced_accuracy` (= competition metric, mean of per-class recall) + `f1_macro` as a secondary check.

In [3]:
# models = {
#     'LogisticRegression': LogisticRegression(
#         max_iter=1000, multi_class='multinomial', n_jobs=-1,
#         random_state=RANDOM_STATE,
#     ),
#     'DecisionTree': DecisionTreeClassifier(
#         max_depth=20, random_state=RANDOM_STATE,
#     ),
#     'RandomForest': RandomForestClassifier(
#         n_estimators=200, max_depth=20, n_jobs=-1,
#         random_state=RANDOM_STATE,
#     ),
#     'HistGradientBoosting': HistGradientBoostingClassifier(
#         max_iter=400, learning_rate=0.05, max_depth=None,
#         random_state=RANDOM_STATE,
#     ),
#     'XGBoost': xgb.XGBClassifier(
#         n_estimators=400, learning_rate=0.05, max_depth=6,
#         subsample=0.9, colsample_bytree=0.9,
#         objective='multi:softprob', num_class=3,
#         eval_metric='mlogloss', tree_method='hist',
#         n_jobs=-1, random_state=RANDOM_STATE,
#     ),
#     'LightGBM': lgb.LGBMClassifier(
#         n_estimators=400, learning_rate=0.05, max_depth=-1, num_leaves=63,
#         subsample=0.9, colsample_bytree=0.9,
#         objective='multiclass', num_class=3,
#         n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
#     ),
#     'CatBoost': cb.CatBoostClassifier(
#         iterations=400, learning_rate=0.05, depth=6,
#         loss_function='MultiClass', random_seed=RANDOM_STATE, verbose=0,
#     ),
#     'Stacking (LGBM+Cat -> LR)': StackingClassifier(
#         estimators=[
#             ('lgb', lgb.LGBMClassifier(
#                 n_estimators=400, learning_rate=0.05, num_leaves=63,
#                 subsample=0.9, colsample_bytree=0.9,
#                 objective='multiclass', num_class=3,
#                 n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
#             )),
#             ('cat', cb.CatBoostClassifier(
#                 iterations=400, learning_rate=0.05, depth=6,
#                 loss_function='MultiClass',
#                 random_seed=RANDOM_STATE, verbose=0,
#             )),
#         ],
#         final_estimator=LogisticRegression(
#             max_iter=1000, multi_class='multinomial',
#             random_state=RANDOM_STATE,
#         ),
#         stack_method='predict_proba',
#         cv=5,
#         n_jobs=1,
#         passthrough=False,
#     ),
#     'Voting (LGBM+Cat, soft)': VotingClassifier(
#         estimators=[
#             ('lgb', lgb.LGBMClassifier(
#                 n_estimators=400, learning_rate=0.05, num_leaves=63,
#                 subsample=0.9, colsample_bytree=0.9,
#                 objective='multiclass', num_class=3,
#                 n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
#             )),
#             ('cat', cb.CatBoostClassifier(
#                 iterations=400, learning_rate=0.05, depth=6,
#                 loss_function='MultiClass',
#                 random_seed=RANDOM_STATE, verbose=0,
#             )),
#         ],
#         voting='soft',
#         n_jobs=1,
#     ),
# }
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# scoring = ['balanced_accuracy', 'f1_macro']

### With Class Weighting

In [ ]:
classes = np.unique(y_train)
cw = compute_class_weight('balanced', classes=classes, y=y_train)

models = {
    'LogisticRegression': LogisticRegression(
        max_iter=1000, multi_class='multinomial', n_jobs=-1,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'DecisionTree': DecisionTreeClassifier(
        max_depth=20, class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, max_depth=20, n_jobs=-1,
        class_weight='balanced',
        random_state=RANDOM_STATE,
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(
        max_iter=400, learning_rate=0.05, max_depth=None,
        class_weight='balanced',                              # sklearn >= 1.4
        random_state=RANDOM_STATE,
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=63,
        subsample=0.9, colsample_bytree=0.9,
        objective='multiclass', num_class=3,
        class_weight='balanced',                              # <-- key line
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
    ),
}
# Add CatBoost separately because of the required .tolist() conversion
models['CatBoost'] = cb.CatBoostClassifier(
    iterations=1000,
            learning_rate=0.09682530687769166,
            depth=4, l2_leaf_reg=6.59689095776599,
            bagging_temperature=0.527738847069491,
            random_strength=0.7128283543743552,
            loss_function='MultiClass',
            random_seed=42,
            early_stopping_rounds=50,
            verbose=0
)

_lgb_base = dict(
    n_estimators=2000, learning_rate=0.05, num_leaves=15,
    subsample=0.8, colsample_bytree=0.9, min_child_samples=100,
    objective='multiclass', num_class=3,
    reg_alpha=0.1, reg_lambda=1.0,
    class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
)
_cat_base = dict(
    iterations=1000,
    learning_rate=0.09682530687769166,
    depth=4, l2_leaf_reg=6.59689095776599,
    bagging_temperature=0.527738847069491,
    random_strength=0.7128283543743552,
    loss_function='MultiClass',
    random_seed=RANDOM_STATE,
    verbose=0,
)
_xgb_base = dict(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    tree_method='hist',
    eval_metric='mlogloss',
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=0,
)

models['Voting (LGBM+Cat+XGB, soft)'] = VotingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**_lgb_base)),
        ('cat', cb.CatBoostClassifier(**_cat_base)),
        ('xgb', xgb.XGBClassifier(**_xgb_base)),
    ],
    voting='soft',
    n_jobs=1,
)

# Stacking sweep: ablate one design choice at a time vs S1 baseline.
# Lighter base learners (n_est=500) keep the sweep tractable; promote the
# winning recipe back to heavier bases for the actual submission.
_n_est_sweep = 500
_lgb_sweep = {**_lgb_base, 'n_estimators': _n_est_sweep}
_cat_sweep = {**_cat_base, 'iterations':   _n_est_sweep}
_xgb_sweep = {**_xgb_base, 'n_estimators': _n_est_sweep}
_rf_sweep = dict(
    n_estimators=300, max_depth=20,
    class_weight='balanced',
    n_jobs=-1, random_state=RANDOM_STATE,
)
_hgb_sweep = dict(
    max_iter=400, learning_rate=0.05, max_depth=None,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)

def _meta_lr():
    return LogisticRegression(
        max_iter=2000, multi_class='multinomial',
        class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE,
    )

def _meta_lgbm():
    return lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=15,
        objective='multiclass', num_class=3,
        class_weight='balanced',
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
    )

_inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def _stack(estimators, final_estimator, passthrough=False):
    return StackingClassifier(
        estimators=estimators,
        final_estimator=final_estimator,
        stack_method='predict_proba',
        cv=_inner_cv,
        n_jobs=1,
        passthrough=passthrough,
    )

# S1 - baseline: 3 boosters -> linear meta
models['Stack S1: LGBM+Cat+XGB -> LR'] = _stack(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**_lgb_sweep)),
        ('cat', cb.CatBoostClassifier(**_cat_sweep)),
        ('xgb', xgb.XGBClassifier(**_xgb_sweep)),
    ],
    final_estimator=_meta_lr(),
)

# S2 - does the meta benefit from seeing the raw features too?
models['Stack S2: LGBM+Cat+XGB -> LR (passthrough)'] = _stack(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**_lgb_sweep)),
        ('cat', cb.CatBoostClassifier(**_cat_sweep)),
        ('xgb', xgb.XGBClassifier(**_xgb_sweep)),
    ],
    final_estimator=_meta_lr(),
    passthrough=True,
)

# S3 - non-linear meta over the same base probs
models['Stack S3: LGBM+Cat+XGB -> LGBM'] = _stack(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**_lgb_sweep)),
        ('cat', cb.CatBoostClassifier(**_cat_sweep)),
        ('xgb', xgb.XGBClassifier(**_xgb_sweep)),
    ],
    final_estimator=_meta_lgbm(),
)

# S4 - add bagging diversity (RF) on top of the booster trio
models['Stack S4: LGBM+Cat+XGB+RF -> LR'] = _stack(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**_lgb_sweep)),
        ('cat', cb.CatBoostClassifier(**_cat_sweep)),
        ('xgb', xgb.XGBClassifier(**_xgb_sweep)),
        ('rf',  RandomForestClassifier(**_rf_sweep)),
    ],
    final_estimator=_meta_lr(),
)

models['Stack S5: LGBM+Cat+RF+HGB -> LR'] = _stack(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**_lgb_sweep)),
        ('cat', cb.CatBoostClassifier(**_cat_sweep)),
        ('rf',  RandomForestClassifier(**_rf_sweep)),
        ('hgb', HistGradientBoostingClassifier(**_hgb_sweep)),
    ],
    final_estimator=_meta_lr(),
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['balanced_accuracy', 'f1_macro']

## 4. Run Cross-Validation and Eval

In [14]:
results = []
eval_models = {
    name: model for name, model in models.items()
    if isinstance(model, StackingClassifier)
}
n_models = len(eval_models)
print(f'Evaluating {n_models} stacking model(s): {list(eval_models)}')

pbar = tqdm(eval_models.items(), total=n_models, desc='Model CV')
for idx, (name, model) in enumerate(pbar, start=1):
    pbar.set_postfix_str(f'running: {name}')
    tqdm.write(f'[{idx}/{n_models}] START  {name}')

    start = time.time()
    
    cv_out = cross_validate(
        model, X_train, y_train,
        cv=cv, scoring=scoring,
        n_jobs=1, return_train_score=False,)

    elapsed = time.time() - start

    row = {
        'model': name,
        'bal_acc_mean':  cv_out['test_balanced_accuracy'].mean(),
        'bal_acc_std':   cv_out['test_balanced_accuracy'].std(),
        'f1m_mean':      cv_out['test_f1_macro'].mean(),
        'f1m_std':       cv_out['test_f1_macro'].std(),
        'fit_time_s':    cv_out['fit_time'].mean(),
        'elapsed_s':     elapsed,
    }
    results.append(row)

    tqdm.write(
        f'[{idx}/{n_models}] DONE   {name:20s} '
        f'| bal_acc={row["bal_acc_mean"]:.4f} ± {row["bal_acc_std"]:.4f} '
        f'| f1_macro={row["f1m_mean"]:.4f} ± {row["f1m_std"]:.4f} '
        f'| {elapsed:6.1f}s'
    )
pbar.close()
print(f'\nAll {n_models} models complete.')

Evaluating 5 stacking model(s): ['Stack S1: LGBM+Cat+XGB -> LR', 'Stack S2: LGBM+Cat+XGB -> LR (passthrough)', 'Stack S3: LGBM+Cat+XGB -> LGBM', 'Stack S4: LGBM+Cat+XGB+RF -> LR', 'Stack S5: LGBM+Cat+RF+HGB -> LR']


[1/5] START  Stack S1: LGBM+Cat+XGB -> LR


[1/5] DONE   Stack S1: LGBM+Cat+XGB -> LR | bal_acc=0.9714 ± 0.0007 | f1_macro=0.9497 ± 0.0014 | 2990.9s
[2/5] START  Stack S2: LGBM+Cat+XGB -> LR (passthrough)


[2/5] DONE   Stack S2: LGBM+Cat+XGB -> LR (passthrough) | bal_acc=0.9690 ± 0.0005 | f1_macro=0.9602 ± 0.0007 | 1549.9s
[3/5] START  Stack S3: LGBM+Cat+XGB -> LGBM


[3/5] DONE   Stack S3: LGBM+Cat+XGB -> LGBM | bal_acc=0.9713 ± 0.0010 | f1_macro=0.9429 ± 0.0016 | 1025.9s
[4/5] START  Stack S4: LGBM+Cat+XGB+RF -> LR


Exception ignored in: <function tqdm.__del__ at 0x10a233640>
Traceback (most recent call last):
  File "/Users/imaadh/Downloads/predicting-irr-kagglecomp/.venv/lib/python3.10/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Users/imaadh/Downloads/predicting-irr-kagglecomp/.venv/lib/python3.10/site-packages/tqdm/std.py", line 1277, in close
    if self.last_print_t < self.start_t + self.delay:
AttributeError: 'tqdm' object has no attribute 'last_print_t'

                                                                                                  
                                                                                                  

[4/5] DONE   Stack S4: LGBM+Cat+XGB+RF -> LR | bal_acc=0.9715 ± 0.0008 | f1_macro=0.9488 ± 0.0011 | 2500.5s
[5/5] START  Stack S5: LGBM+Cat+RF+HGB -> LR



Model CV: 100%|██████████| 5/5 [3:01:45<00:00, 2181.08s/it, running: Stack S5: LGBM+Cat+RF+HGB -> LR]

[5/5] DONE   Stack S5: LGBM+Cat+RF+HGB -> LR | bal_acc=0.9715 ± 0.0008 | f1_macro=0.9486 ± 0.0012 | 2838.0s

All 5 models complete.


## 5. Leaderboard

In [15]:
leaderboard = (
    pd.DataFrame(results)
      .sort_values('bal_acc_mean', ascending=False)
      .reset_index(drop=True)
)
leaderboard

,model,bal_acc_mean,bal_acc_std,f1m_mean,f1m_std,fit_time_s,elapsed_s
0,Stack S5: LGBM+Cat+RF+HGB -> LR,0.971519,0.000819,0.948629,0.001194,563.653351,2838.019644
1,Stack S4: LGBM+Cat+XGB+RF -> LR,0.971511,0.000778,0.948794,0.001107,497.926647,2500.506665
2,Stack S1: LGBM+Cat+XGB -> LR,0.971419,0.000710,0.949733,0.001394,596.790485,2990.913867
3,Stack S3: LGBM+Cat+XGB -> LGBM,0.971278,0.001032,0.942896,0.001555,203.331331,1025.852484
4,Stack S2: LGBM+Cat+XGB -> LR (passthrough),0.969042,0.000450,0.960189,0.000668,308.529154,1549.930074


## 6. Fit best model on full train set & save

The model with the highest mean `balanced_accuracy` (the competition metric) is refit on all of `X_train` and saved to `models/` along with the CV leaderboard.

In [16]:
import re

best_name = leaderboard.sort_values('bal_acc_mean', ascending=False).iloc[0]['model']
best_model = models[best_name]
# Shell-safe filename slug (avoids spaces, parens, '>' etc. that break zsh/bash)
safe_name = re.sub(r'[^A-Za-z0-9._+-]+', '_', best_name).strip('_')
print(f'Refitting best model: {best_name}  (slug: {safe_name})')

best_model.fit(X_train, y_train)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / f'{DATA_VERSION}__{safe_name}.pkl'
lb_path    = MODELS_DIR / f'{DATA_VERSION}__cv_leaderboard_classbalancing.csv'
pickle.dump(best_model, open(model_path, 'wb'))
leaderboard.to_csv(lb_path, index=False)

print(f'Saved model      -> {model_path}')
print(f'Saved leaderboard-> {lb_path}')

Refitting best model: Stack S5: LGBM+Cat+RF+HGB -> LR  (slug: Stack_S5_LGBM+Cat+RF+HGB_-_LR)
Saved model      -> /Users/imaadh/Downloads/predicting-irr-kagglecomp/models/preprocessed_v1.3__Stack_S5_LGBM+Cat+RF+HGB_-_LR.pkl
Saved leaderboard-> /Users/imaadh/Downloads/predicting-irr-kagglecomp/models/preprocessed_v1.3__cv_leaderboard_classbalancing.csv


## 7. Predict on test set

In [17]:
inv_label = {v: k for k, v in LABEL_MAP.items()}
y_pred = best_model.predict(X_test).ravel()
y_pred_labels = pd.Series(y_pred).map(inv_label)

test_ids = pd.read_csv(RAW_DATA_DIR / 'test.csv', usecols=['id'])['id']
assert len(test_ids) == len(y_pred_labels), (
    f'Row mismatch: test.csv has {len(test_ids)} rows but predictions has {len(y_pred_labels)}'
)
submission = pd.DataFrame({
    'id': test_ids.values,
    'Irrigation_Need': y_pred_labels.values,
})
sub_path = MODELS_DIR / f'submission__{DATA_VERSION}__{safe_name}.csv'
submission.to_csv(sub_path, index=False)
print(f'Submission saved -> {sub_path}')
submission.head()

Submission saved -> /Users/imaadh/Downloads/predicting-irr-kagglecomp/models/submission__preprocessed_v1.3__Stack_S5_LGBM+Cat+RF+HGB_-_LR.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
